In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [32]:


RAW_PATH = Path("../data/raw")

PROCESSED_PATH = Path("../data/processed/new_processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Saving files to:")
print(PROCESSED_PATH.resolve())

Saving files to:
C:\Users\vidha\OneDrive\Desktop\bluestock_mf_capstone\data\processed\new_processed


In [4]:
fund_master = pd.read_csv(RAW_PATH / "01_fund_master.csv")
nav_history = pd.read_csv(RAW_PATH / "02_nav_history.csv")
aum = pd.read_csv(RAW_PATH / "03_aum_by_fund_house.csv")
sip = pd.read_csv(RAW_PATH / "04_monthly_sip_inflows.csv")
category = pd.read_csv(RAW_PATH / "05_category_inflows.csv")
folio = pd.read_csv(RAW_PATH / "06_industry_folio_count.csv")
performance = pd.read_csv(RAW_PATH / "07_scheme_performance.csv")
transactions = pd.read_csv(RAW_PATH / "08_investor_transactions.csv")
holdings = pd.read_csv(RAW_PATH / "09_portfolio_holdings.csv")
benchmark = pd.read_csv(RAW_PATH / "10_benchmark_indices.csv")

In [5]:
datasets = {
    "fund_master": fund_master,
    "nav_history": nav_history,
    "aum": aum,
    "sip": sip,
    "category": category,
    "folio": folio,
    "performance": performance,
    "transactions": transactions,
    "holdings": holdings,
    "benchmark": benchmark
}

for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("Shape:", df.shape)
    print(df.head())

fund_master
Shape: (40, 15)
   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular Plan - Growth   

  category sub_category     plan launch_date                  benchmark  \
0   Equity    Large Cap  Regular  2006-02-14              NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01              NIFTY 100 TRI   
2   Equity    Small Cap  Regular  2009-09-09       BSE 250 SmallCap TRI   
3   Equity    Small Cap   Direct  2013-01-01       BSE 250 SmallCap TRI   
4     Debt         Gilt  Regular  2000-12-30  CRISIL Dynamic Gilt Index   

   expense_ratio_pct  exit_load_pct  min_sip_a

In [6]:
nav_history.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB


In [7]:
nav_history["date"] = pd.to_datetime(
    nav_history["date"],
    errors="coerce"
)

In [10]:
nav_history = nav_history.dropna(subset=["date"])

In [11]:
nav_history = nav_history.sort_values(
    ["amfi_code", "date"]
)

In [9]:
before = len(nav_history)

nav_history = nav_history.drop_duplicates()

after = len(nav_history)

print("Duplicates Removed:", before - after)

Duplicates Removed: 0


In [12]:
nav_history = nav_history[
    nav_history["nav"] > 0
]

In [13]:
nav_history["nav"] = nav_history.groupby(
    "amfi_code"
)["nav"].ffill()

In [14]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [15]:
transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.title()
)

transactions["transaction_type"] = (
    transactions["transaction_type"]
    .replace({
        "Sip": "SIP",
        "Lumpsum": "Lumpsum",
        "Redemption": "Redemption"
    })
)

In [16]:
transactions = transactions[
    transactions["amount_inr"] > 0
]

In [17]:
transactions = transactions.drop_duplicates()

In [18]:
performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [19]:
numeric_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct"
]

for col in numeric_cols:
    if col in performance.columns:
        performance[col] = pd.to_numeric(
            performance[col],
            errors="coerce"
        )

In [20]:
performance = performance.dropna()

In [21]:
if "expense_ratio_pct" in performance.columns:
    performance = performance[
        performance["expense_ratio_pct"]
        .between(0.1, 2.5)
    ]
    

In [22]:
negative_sharpe = performance[
    performance["sharpe_ratio"] < 0
]

print("Funds with Negative Sharpe Ratio")
print(negative_sharpe.shape)

Funds with Negative Sharpe Ratio
(0, 19)


In [23]:
fund_master = fund_master.drop_duplicates()

In [24]:
if "launch_date" in fund_master.columns:
    fund_master["launch_date"] = pd.to_datetime(
        fund_master["launch_date"],
        errors="coerce"
    )

In [25]:
aum = aum.drop_duplicates()
aum = aum.fillna(0)

In [26]:
sip = sip.drop_duplicates()
sip = sip.fillna(0)

In [27]:
category = category.drop_duplicates()
category = category.fillna(0)

In [28]:
folio = folio.drop_duplicates()
folio = folio.fillna(0)

In [29]:
holdings = holdings.drop_duplicates()
holdings = holdings.fillna(0)

In [30]:
if "date" in benchmark.columns:
    benchmark["date"] = pd.to_datetime(
        benchmark["date"],
        errors="coerce"
    )

In [31]:
datasets = {
    "fund_master": fund_master,
    "nav_history": nav_history,
    "aum": aum,
    "sip": sip,
    "category": category,
    "folio": folio,
    "performance": performance,
    "transactions": transactions,
    "holdings": holdings,
    "benchmark": benchmark
}

for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Missing Values:", df.isnull().sum().sum())

fund_master
Rows: 40
Columns: 15
Missing Values: 0
nav_history
Rows: 46000
Columns: 3
Missing Values: 0
aum
Rows: 90
Columns: 5
Missing Values: 0
sip
Rows: 48
Columns: 6
Missing Values: 0
category
Rows: 144
Columns: 3
Missing Values: 0
folio
Rows: 21
Columns: 6
Missing Values: 0
performance
Rows: 40
Columns: 19
Missing Values: 0
transactions
Rows: 32778
Columns: 13
Missing Values: 0
holdings
Rows: 322
Columns: 8
Missing Values: 0
benchmark
Rows: 8050
Columns: 3
Missing Values: 0


In [33]:
fund_master.to_csv(
    PROCESSED_PATH / "clean_fund_master.csv",
    index=False
)

nav_history.to_csv(
    PROCESSED_PATH / "clean_nav_history.csv",
    index=False
)

aum.to_csv(
    PROCESSED_PATH / "clean_aum.csv",
    index=False
)

sip.to_csv(
    PROCESSED_PATH / "clean_sip.csv",
    index=False
)

category.to_csv(
    PROCESSED_PATH / "clean_category.csv",
    index=False
)

folio.to_csv(
    PROCESSED_PATH / "clean_folio.csv",
    index=False
)

performance.to_csv(
    PROCESSED_PATH / "clean_performance.csv",
    index=False
)

transactions.to_csv(
    PROCESSED_PATH / "clean_transactions.csv",
    index=False
)

holdings.to_csv(
    PROCESSED_PATH / "clean_holdings.csv",
    index=False
)

benchmark.to_csv(
    PROCESSED_PATH / "clean_benchmark.csv",
    index=False
)

print("All cleaned files saved successfully.")

All cleaned files saved successfully.


In [34]:
for file in PROCESSED_PATH.glob("*.csv"):
    print(file.name)

clean_aum.csv
clean_benchmark.csv
clean_category.csv
clean_folio.csv
clean_fund_master.csv
clean_holdings.csv
clean_nav_history.csv
clean_performance.csv
clean_sip.csv
clean_transactions.csv
